# 02 — Stage and Aggregate Weather

**Pipeline stage:** ERA5 weather → per-country weather series, at a chosen spatial resolution and
aggregation scheme.

Three steps, each a separate script:

1. `stage_era5_arco.py` — pulls the three raw ERA5 fields we need (100 m wind components, 2 m
   temperature, surface solar radiation) for one country's bounding box, at native 0.25° resolution.
2. `prepare_era5_resolution_cache.py` — pre-computes coarsened versions of that native grid at
   0.5°, 1°, and 2°, so later experiments don't re-coarsen from scratch every time.
3. `weather_informed/weather_build.py` (formerly `build_weighted_weather_local.py`) — combines a
   chosen resolution's weather grid with a capacity map (from notebook 01) to produce one hourly
   weather series per country, either uniformly averaged or capacity-weighted.

Staging needs a Copernicus Climate Data Store key (`CDSAPI_KEY` in `.env`, or `~/.cdsapirc`); nothing
else in this notebook does.

## The three weather variables

100 m wind speed isn't in ERA5 directly — it's derived from the eastward/northward wind *components*,
`u100` and `v100`, via `V = sqrt(u^2 + v^2)` (the paper's wind-speed equation). Solar radiation is
converted from accumulated energy (J/m²) to power (W/m²) by dividing by 3600, and temperature from
kelvin to Celsius. Here's the wind-speed step for real, on toy component values:

In [1]:
import numpy as np

# Toy u/v wind components (m/s) for three made-up hours.
u100 = np.array([3.0, -2.0, 0.0])
v100 = np.array([4.0, 0.0, 5.0])

wind_speed_100m = np.sqrt(u100**2 + v100**2)
print("100 m wind speed:", wind_speed_100m, "m/s")
assert np.allclose(wind_speed_100m, [5.0, 2.0, 5.0])

100 m wind speed: [5. 2. 5.] m/s


## Spatial resolution: coarsening by block-averaging

Each coarser resolution is built by averaging contiguous, non-overlapping blocks of the native 0.25°
grid: a 0.5° cell is a 2×2 block (4 native cells), a 1° cell is 4×4 (16 native cells), and a 2° cell is
8×8 (64 native cells). This is why coarsening cuts the number of processed cells so sharply — a toy
4×4 native grid collapsing to a single 1° cell:

In [2]:
import numpy as np

rng = np.random.default_rng(42)
native_0p25deg = rng.uniform(5, 9, size=(4, 4))  # a toy 4x4 block of wind-speed values
print("Native 0.25 deg block (16 cells):\n", np.round(native_0p25deg, 2))

coarsened_1deg = native_0p25deg.mean()
print(f"\nCoarsened to a single 1 deg cell: {coarsened_1deg:.2f} m/s (mean of all 16)")
print(f"Cell count: 16 -> 1, a {100 * (1 - 1/16):.1f}% reduction for this one cell.")

Native 0.25 deg block (16 cells):
 [[8.1  6.76 8.43 7.79]
 [5.38 8.9  8.04 8.14]
 [5.51 6.8  6.48 8.71]
 [7.58 8.29 6.77 5.91]]

Coarsened to a single 1 deg cell: 7.35 m/s (mean of all 16)
Cell count: 16 -> 1, a 93.8% reduction for this one cell.


In [3]:
!python ../scripts/stage_era5_arco.py --help

usage: stage_era5_arco.py [-h] --codes
                          {at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx}
                          [{at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} ...]
                          [--start START] [--end END] [--padding PADDING]
                          [--out-dir OUT_DIR] [--results-dir RESULTS_DIR]
                          [--force] [--dry-run]

options:
  -h, --help            show this help message and exit
  --codes {at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} [{at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} ...]
  --start START
  --end END
  --padding PADDING
  --out-dir OUT_DIR
  --results-dir RESULTS_DIR
  --force
  --dry-run


In [4]:
!python ../scripts/prepare_era5_resolution_cache.py --help

usage: prepare_era5_resolution_cache.py [-h] --codes
                                        {dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro,tx}
                                        [{dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro,tx} ...]
                                        [--resolutions RESOLUTIONS [RESOLUTIONS ...]]
                                        [--start START] [--end END]
                                        [--era5-dir ERA5_DIR]
                                        [--out-dir OUT_DIR]
                                        [--results-dir RESULTS_DIR] [--force]

options:
  -h, --help            show this help message and exit
  --codes {dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro,tx} [{dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro,tx} ...]
  --resolutions RESOLUTIONS [RESOLUTIONS ...]
  --start START
  --end END
  --era5-dir ERA5_DIR
  --out-dir OUT_DIR
  --results-dir RESULTS_DIR
  --force


**DATA CELL — not run here.** Staging one country's native grid, then building the four resolution
caches, looks like:

```bash
python scripts/stage_era5_arco.py --codes de --start 2022-01-01 --end 2026-04-30
python scripts/prepare_era5_resolution_cache.py --codes de --resolutions 0.25 0.5 1.0 2.0
```

Both are resumable — re-running with the same arguments skips work that's already on disk, unless
`--force` is passed.

## Aggregating weather into one national series (`weather_informed/weather_build.py`)

This is the module that actually applies the capacity-weighting idea from notebook 01 to a real
weather grid: it maps GEM facility points to their nearest grid cell, sums technology-specific capacity
per cell, and produces a capacity-weighted (or, with `--scheme uniform`, a plain-averaged) national
hourly series for wind speed, solar radiation, and temperature. It also has its own command-line
interface, so it can be run standalone per country/resolution/scheme rather than only through the
cache-preparation step above.

In [5]:
!python ../weather_informed/weather_build.py --help

usage: weather_build.py [-h] --code CODE --resolution RESOLUTION
                        [--scheme {capacity,uniform}] [--start START]
                        [--end END] [--era5-dir ERA5_DIR]
                        [--capacity-dir CAPACITY_DIR] [--out-dir OUT_DIR]
                        [--resolution-cache-dir RESOLUTION_CACHE_DIR]
                        [--no-resolution-cache] [--output OUTPUT]
                        [--metrics-json METRICS_JSON]

options:
  -h, --help            show this help message and exit
  --code CODE
  --resolution RESOLUTION
  --scheme {capacity,uniform}
  --start START
  --end END
  --era5-dir ERA5_DIR
  --capacity-dir CAPACITY_DIR
  --out-dir OUT_DIR
  --resolution-cache-dir RESOLUTION_CACHE_DIR
  --no-resolution-cache
  --output OUTPUT
  --metrics-json METRICS_JSON


**DATA CELL — not run here.** A real build for one country/resolution/scheme:

```bash
python weather_informed/weather_build.py --code de --resolution 1.0 --scheme capacity
```

This writes `data/country_weather_post_covid/weather_era5_de_capacity_1.0deg.csv`.

**Next:** [03 — Evaluate Calendar vs. Weather Models](03_evaluate_calendar_vs_weather_models.ipynb)
takes one of these weather series, joins it to the electricity target from notebook 01, and measures
how much predictive value the weather actually adds.